# Rag from Scracth

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# LANGCHAIN_TRACING_V2: enables tracing of runs in LangSmith
# LANGSMITH_ENDPOINT: URL of the LangSmith service traces are sent to (depends on the ubication)
# LANGSMITH_API_KEY: credential used to authenticate traces in LangSmith
# OPENAI_API_KEY: credential used to authenticate requests to the OpenAI API
# (used for embeddings and/or chat completions in the RAG pipeline)

# Part 2: Query Translations
Query transformation refers to techniques that rephrase or adjust a question so it works better for retrieval.

## Part 2.1: Multi Query

### Index

In [ ]:
### Indexing

import bs4
from langchain_community.document_loaders import WebBaseLoader

# Download a web article
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Coffee",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(id=["firstHeading", "mw-content-text"])
    ),
)
blog_docs = loader.load()

# Split it in chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
  chunk_size=1000, # Each chunk 1000 characters
  chunk_overlap=200 # Each chunk repeat the 200 last characters of the previous one
)

splits = text_splitter.split_documents(blog_docs)

# Create a vector database from the chunks
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
  documents=splits, # Use the digested chunks as documents
  embedding=OpenAIEmbeddings() # Use OpenAI embeddings to vectorize the documents
)

retriever = vectorstore.as_retriever(search_kwargs={"k":10})

### Prompting

The prompt asks the LLM to rewrite the user's question in five different ways, one per line, to improve retrieval.
The chain fills the template, calls the model, extracts the text and splits it into a list of five queries.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# First we have to generate the different perspectives of the question,
# so we can retrieve more relevant passages from the vector database
template = """You are a helpful assistant that improves document search. Rewrite the question below in {number}
different ways, each one approaching the topic from a distinct angle or using different wording.
The goal is to retrieve as many relevant passages as possible from a vector database,
even when the original phrasing does not match the text closely. Return only the {number} rewritten questions,
one per line, with no numbering and no extra commentary. Question: {question}"""

# Transform the template into a prompt that can be used with a language model
prompt_perspectives = ChatPromptTemplate.from_template(template)

# Generate the different perspectives of the question

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# The chain is built with LCEL: each "|" pipes the output of one step into the next
generate_queries = (
    prompt_perspectives # Fills {question} in the template and builds the chat messages for the model
    | ChatOpenAI(model_name="gpt-5-nano", temperature=0) # Calls the LLM; temperature=0 makes the output 0% random. Returns an AIMessage
    | StrOutputParser() # Extracts the plain text (str) from the AIMessage
    | (lambda x: x.split("\n")) # Splits the text into a list of strings, one per line (relies on the prompt asking for one question per line)
)

In this example, as it can be seen at LangSmith console, the 5 questions generated by OpenAI LLM are the following:
- What environmental impacts and social issues are associated with coffee production?
- What ecological effects and community-level challenges accompany coffee farming?
- Which environmental risks and labor-related or social consequences are linked to the coffee industry?
- What impacts on ecosystems and local populations arise from coffee cultivation and processing?
- What sustainability concerns—environmental and social—are connected to the coffee supply chain?

Now, having the questions, the next step is asking them to the LLM and join the answers

In [ ]:
from langchain_core.load import dumps, loads

# Receives a list of lists, where each list contains the questions, the documents retrieved.
def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Delete duplicates by converting to a set and back to a list
    unique_docs = list(set(flattened_docs))
    # Return in Document format again
    return [loads(doc) for doc in unique_docs]

# Retrieve
# Users possible question
question = "What are the environmental and social problems linked to coffee production?"

# Retrieval chain with Multi Query:
# 1) generate_queries: the LLM rewrites the question in 5 different ways
# 2) retriever.map(): searches Chroma with EACH rewrite (list of lists of chunks)
# 3) get_unique_union: flattens the result and removes duplicate chunks
retrieval_chain = generate_queries | retriever.map() | get_unique_union

# Executes the retrieval chain with the user question, and returns the retrieved documents
docs = retrieval_chain.invoke({"question":question, "number":5})
len(docs)

The last part is to take retrieved infromation by the model to each one of the questions

In [ ]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# Call the model
llm = ChatOpenAI(
  model_name="gpt-5-nano", # NOV 2026 the more affordable OpenAI model is GPT 5 Nano
  temperature=0 # Want the model to be 0 creative
)


final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question, "number":5})

## Part 2.2: RAG Fusion

### Prompt

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# The following prompt is used to generate multiple search queries based on a single input query
template = """You are a helpful assistant that improves document search. Rewrite the question below in {number}
different ways, each one approaching the topic from a distinct angle or using different wording.
The goal is to retrieve as many relevant passages as possible from a vector database,
even when the original phrasing does not match the text closely. Return only the {number} rewritten questions,
one per line, with no numbering and no extra commentary. Question: {question}"""

# Transform the template into a prompt that can be used with a language model
prompt_ragfusion = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Same as Multi Query, but with a different prompt
generate_queries = (
    prompt_ragfusion # Fills {question} in the template and builds the chat messages for the model
    | ChatOpenAI(model_name="gpt-5-nano", temperature=0) # Calls the LLM; temperature=0 makes the output 0% random. Returns an AIMessage
    | StrOutputParser() # Extracts the plain text (str) from the AIMessage
    | (lambda x: x.split("\n")) # Splits the text into a list of strings, one per line (relies on the prompt asking for one question per line)
)

Once we have the different outputs, it is time to combine multiple lists of retrieved documents into a single ranking, giving more weight to those that appear in high positions across several lists.

In [ ]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ results is a list of lists, each list contains the ranked documents retrieved for each query.
      k is a optional parameter used in the RRF formula. The function returns a list of unique documents ranked by their RRF score. """

    # Dict to store the RRF scores for each document
    scores = {}


    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through document in each query's results, being rank de position of the document in the list
        for rank, doc in enumerate(docs):

          # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
          doc_str = dumps(doc)

          # If the document is not yet in the dictionary, add it with an initial score of 0
          if doc_str not in scores:
            scores[doc_str] = 0

          # Update the RRF score for the document using the formula: 1 / (k + rank)
          # Update the score of the document using the RRF formula: 1 / (rank + k)
          scores[doc_str] += 1 / (rank + k)

    # Sort the documents by their RRF scores in descending order
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results


# Final chain: first we generate multiple queries, then we retrieve the documents for each query, and finally we fuse the results using RRF
rag_fusion_chain = (
  generate_queries
  | retriever.map()
  | reciprocal_rank_fusion
)

docs = rag_fusion_chain.invoke({"question": question, "number":10})
len(docs)

The line `scores[doc_str] += 1 / (rank + k)` is the core of **Reciprocal Rank Fusion (RRF)**. Each time a document shows up in one of the ranked lists (one list per rewritten query), it earns a score of `1 / (rank + k)`, and the scores from all the lists are summed.

- **`rank`**: position of the document in that list (0 = best). The higher it is ranked, the bigger its contribution.
- **`k`** (default 60): smoothing constant. It stops the top positions from dominating too much, so a document ranked 1st in a single list does not automatically beat one that is consistently ranked 3rd-5th in several lists.
- **Summing across lists**: documents retrieved by many different query rewrites accumulate a higher score, so they end up at the top of the final ranking.

The result is a single ranking that favours documents that are both highly ranked and repeatedly retrieved, without needing to compare the raw similarity scores of different queries (which are not directly comparable).

Once we have ranked and fused documents, we use a LLM as a context in order to retrieve the final answer

In [ ]:

from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": rag_fusion_chain, # Use as a context the result of invoke of the RAG Fusion chain
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question, "number":10})

### RAG Fusion: summary

**`final_rag_chain`**
1. **`rag_fusion_chain`** → builds the `{context}`
   1. **`generate_queries`**: `prompt_ragfusion` → LLM → `StrOutputParser` → `split("\n")` (list of rewritten questions)
   2. **`retriever.map()`**: searches Chroma with each rewritten question (list of lists of chunks)
   3. **`reciprocal_rank_fusion`**: sums `1 / (rank + k)` per chunk and sorts by score (single ranking)
2. **`itemgetter("question")`** → keeps the original `{question}`
3. **`prompt`** → fills `{context}` and `{question}`
4. **`llm`** → generates the answer
5. **`StrOutputParser`** → returns the answer as plain text

## Part 2.3: Decomposition

**Decomposition prompt**: Tell LLM to convert a simple question into a different simpler questions. In this part we just only prepare the template.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# Prompt to generate multiple sub-questions related to an input question
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output ({number} queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

# LLM to generate multiple sub-questions related to an input question
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)

# Chain to generate multiple sub-questions related to an input question
generate_different_queries_chain = (
  prompt_decomposition
  | llm
  | StrOutputParser()
  | (lambda x: x.split("\n"))
  )

# Run the chain
question = "Why does Brazil dominate world coffee production and what environmental and social consequences does this have?"
sub_questions = generate_different_queries_chain.invoke({"question":question, "number":3})

In [ ]:
sub_questions

### **Part 2.3.1: Recursive form**

The first step is to define the prompt that answers each sub-question, using the previous questions and their answers as context. The variables are:

- `{question}`: the sub-question to answer now. It appears twice: at the beginning, to introduce it, and at the end, to remind the model what it has to answer after reading all the context.
- `{q_a_pairs}`: the question-answer pairs from the previous sub-questions. This corresponds to the green arrow in the slide, where the result for "think, machine" → "ke" is reused to solve "think, machine, learning". In the first iteration it is empty, and in the following ones it accumulates the previous answers.
- `{context}`: the documents retrieved from the vector store for the current sub-question (the RAG retrieval).

The final instruction, *"Use the above context and any background question + answer pairs to answer the question"*, tells the model to combine both sources of information: what was retrieved from the documents and what was learned in previous steps.

Separators are just to meake easier to the model where question ends and context starts.

In [ ]:
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

Now, the loop to iterate through different sub-questions

In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

# Format to each question - answer pair as a string, and join them with a separator
def format_qa(question, answer):
	formatted_string = ""
	formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
	return formatted_string.strip()

# LLM to answer the question based on the context and any background question + answer pairs
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)

q_a_pairs = ""

# Iterate through sub questions
for q in sub_questions:
	chain = (
		{"context": itemgetter("question") | retriever, # Use the retriever to get the context for each sub-question
     "question": itemgetter("question"), # Use the sub-question as the question to answer
     "q_a_pairs": itemgetter("q_a_pairs")} # Use the previous question + answer pairs as context for the next sub-question

		 | decomposition_prompt # Take the context, question, and previous question + answer pairs and format them into a prompt for the LLM
		 | llm
		 | StrOutputParser() # Extract the answer from the LLM's output
	)

	answer = chain.invoke({"question": q, "q_a_pairs": q_a_pairs}) # Execute chain with actual pairs
	q_a_pair = format_qa(q, answer)
	q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair # Add the new question + answer pair to the context for the next sub-question

In [ ]:
answer

*It is useful to se in LangChain console the questions and answer to every OpenAI model call, in order to appreciate the q-a pairs*

### **Part 2.3.2: Answer individually**

It is possible to answer every question individually and then use an llm to answer a general question based on those different answers.
This option may fit better in cases when question is not well divissible in many sub-questions

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


# Local copy of the "rlm/rag-prompt" template from the LangChain Hub.
# Defined here instead of pulled via Client().pull_prompt(...) because the Hub's public
# prompts live only in the US region and 404 from an EU-region LangSmith account.
prompt = ChatPromptTemplate.from_messages([
    ("human", """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.
Question: {question}
Context: {context}
Answer:"""),
])

def retrieve_and_rag(question,number, prompt_rag,sub_question_generator_chain):

	# First we generate multiple sub-questions related to the input question
	sub_questions = sub_question_generator_chain.invoke({"question":question, "number":number})

	# List that contains RAG chain results
	rag_results = []

	# Iterate through sub questions
	for q in sub_questions:

		# Retrieve the context for each sub-question
		retrieved_documents = retriever.invoke(q)

		# Mini chain: prompt --> LLM --> String
		mini_chain = (prompt_rag | llm | StrOutputParser())
		answer = mini_chain.invoke({"question": q, "context": retrieved_documents})

		# Append the question and answer pair to the results list
		rag_results.append((q, answer))

	return rag_results, sub_questions

# Execute with the original question and extract two paralel lists: the RAG results and the sub-questions
answers, sub_questions = retrieve_and_rag(question, 3, prompt, generate_different_queries_chain)

Having a list with al questions and answers, it is time to ask LLM to make a final answer based on that list as the context

In [ ]:
def format_qa_pairs(questions, answers):
    """Make easier to LLM to understand the list, as the input has 'Question i: Answer i: ...' format."""
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(sub_questions, answers)

# Prompt to explain LLM
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})